# 7.1. 하이브리드 추천 시스템의 장점

실제 운영환경에서는 다수의 알고리즘을 결합해서 **하이브리드 알고리즘** 을 사용하는 경우가 많음.

그 이유는, 다수의 알고리즘을 결합하는 것이 단일 알고리즘을 사용하는 것보다 더 정확한 경우가 많기 때문. 

다수의 알고리즘이 결합되어 평균을 내는 형태가 아니라, 상호보완을 하는 역할을 수행함. 물론, 하이브리드 알고리즘이 항상 좋은 결과를 내는 것은 아니기 때문에, 다양한 실험을 통한 결정이 필요함.

# 7.2. 하이브리드 추천 시스템의 원리

In [2]:
from sklearn.model_selection import train_test_split
import random
import numpy as np
import pandas as pd

r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv('../data/u.data',
                      names=r_cols,
                      sep='\t',
                      encoding='latin-1')

ratings_train,ratings_test = train_test_split(ratings,
                                              test_size=0.2,
                                              shuffle=True,
                                              random_state=2021)

def RMSE2(y_true,y_pred):
  return np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2))


def recommender_1(recom_list):
  recommendations = []
  for pair in recom_list:
    recommendations.append(random.random() * 4 +1)
  return np.array(recommendations)

def recommender_2(recom_list):
  recommendations = []
  for pair in recom_list:
    recommendations.append(random.random() * 4 +1)
  return np.array(recommendations)


weight = [0.8,0.2]
recom_list = np.array(ratings_test)
predictions_1 = recommender_1(recom_list)
predictions_2 = recommender_2(recom_list)

predictions = predictions_1 * weight[0] + predictions_2 * weight[1]
RMSE2(recom_list[:,2],predictions)

1.56266993908199

# 7.3. 하이브리드 추천 시스템 (CF+MF)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import os
import numpy as np
import pandas as pd

class NEW_MF():
  def __init__(self,ratings,hyper_params):
    self.R = np.array(ratings)
    self.num_users,self.num_items = np.shape(self.R)

    self.K = hyper_params['K']
    self.alpha = hyper_params['alpha']
    self.beta = hyper_params['beta']
    self.iterations = hyper_params['iterations']
    self.verbose = hyper_params['verbose']

    item_id_index = []
    index_item_id = []
    for i, one_id in enumerate(ratings):
      item_id_index.append([one_id,i])
      index_item_id.append([i,one_id])
    self.item_id_index = dict(item_id_index)
    self.index_item_id = dict(index_item_id)

    user_id_index = []
    index_user_id = []
    for i, one_id in enumerate(ratings.T):
      user_id_index.append([one_id,i])
      index_user_id.append([i,one_id])
    self.user_id_index = dict(user_id_index)
    self.index_user_id = dict(index_user_id)

  def rmse(self):
    xs, ys = self.R.nonzero()
    self.predictions = []
    self.errors = []
    for x,y in zip(xs,ys):
      prediction = self.get_prediction(x,y)
      self.predictions.append(prediction)
      self.errors.append(self.R[x,y] - prediction)
    self.predictions = np.array(self.predictions)
    self.errors = np.array(self.errors)
    return np.sqrt(np.mean(self.errors**2))

  def sgd(self):
    for i,j,r in self.samples:
      prediction = self.get_prediction(i,j)
      e = (r - prediction)

      self.b_u[i] += self.alpha * (e - (self.beta * self.b_u[i]))
      self.b_d[j] += self.alpha * (e - (self.beta * self.b_d[j]))

      self.P[i,:] += self.alpha * ((e * self.Q[j,:]) - (self.beta * self.P[i,:]))
      self.Q[j,:] += self.alpha * ((e * self.P[i,:]) - (self.beta * self.Q[j,:]))

  def get_prediction(self,i,j):
    prediction = self.b + self.b_u[i] + self.b_d[j] + self.P[i,:].dot(self.Q[j,:].T)
    return prediction

  def set_test(self,ratings_test):
    test_set = []
    for i in range(len(ratings_test)):
      x = self.user_id_index[ratings_test.iloc[i,0]]
      y = self.item_id_index[ratings_test.iloc[i,1]]
      z = ratings_test.iloc[i,2]
      test_set.append([x,y,z])
      self.R[x,y] = 0
    self.test_set = test_set
    return test_set
 
  def test_rmse(self):
    error = 0
    for one_set in self.test_set:
      predicted = self.get_prediction(one_set[0],one_set[1])
      error += pow(one_set[2] - predicted,2)
    return np.sqrt(error/len(self.test_set))
 
  def test(self):
    self.P = np.random.normal(scale=1./self.K,
                              size=(self.num_users,self.K))
    self.Q = np.random.normal(scale=1./self.K,
                              size=(self.num_items,self.K))
    self.b_u = np.zeros(self.num_users)
    self.b_d = np.zeros(self.num_items)
    self.b = np.mean(self.R[self.R.nonzero()])

    rows, columns = self.R.nonzero()
    self.samples = [(i,j,self.R[i,j]) for i,j in zip(rows, columns)]

    training_process = []
    for i in range(self.iterations):
      np.random.shuffle(self.samples)
      self.sgd()
      rmse1 = self.rmse()
      rmse2 = self.test_rmse()
      training_process.append((i+1,rmse1,rmse2))
      if self.verbose:
        if (i+1) % 10 == 0:
          print("Iteration : %d ; Train RMSE = %.4f ; Test RMSE = %.4f" % (i+1,rmse1,rmse2))
   
    return training_process

  def get_one_prediction(self,user_id,item_id):
    return self.get_prediction(self.user_id_index[user_id],
                               self.item_id_index[item_id])

  def full_prediction(self):
    return self.b + self.b_u[:,np.newaxis] + self.b_d[np.newaxis,:] + self.P.dot(self.Q.T)

base_src = '../data/'
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')

R_temp = ratings.pivot(index='user_id',
                       columns='movie_id',
                       values='rating').fillna(0)


ratings_train,ratings_test = train_test_split(ratings,
                                              test_size=0.2,
                                              shuffle=True,
                                              random_state=2021)

hyper_params = {
    "K":30,
    "alpha":0.001,
    "beta":0.02,
    "iterations":100,
    "verbose":True
}

mf = NEW_MF(R_temp,hyper_params)
test_set = mf.set_test(ratings_test)
result = mf.test()

######################################

rating_matrix = ratings_train.pivot(index='user_id',columns = 'movie_id',values='rating')

rating_mean = rating_matrix.mean(axis=1)
rating_bias = (rating_matrix.T - rating_mean).T

matrix_dummy = rating_matrix.copy().fillna(0)
user_similarity = cosine_similarity(matrix_dummy,matrix_dummy)
user_similarity = pd.DataFrame(user_similarity,
                               index=rating_matrix.index,
                               columns=rating_matrix.index)

def CF_knn_bias(user_id,movie_id,neighbor_size=0):
  if movie_id in rating_bias.columns:
    sim_scores = user_similarity[user_id].copy()
    movie_ratings = rating_bias[movie_id].copy()
    none_rating_idx = movie_ratings[movie_ratings.isnull()].index
    movie_ratings = movie_ratings.drop(none_rating_idx)
    sim_scores = sim_scores.drop(none_rating_idx)

    if neighbor_size == 0:
      prediction = np.dot(sim_scores,movie_ratings) / sim_scores.sum()
      prediction = prediction + rating_mean[user_id]
  
    else:
      if len(sim_scores) > 1:
        neighbor_size = min(neighbor_size,len(sim_scores))
        sim_scores = np.array(sim_scores)
        movie_ratings = np.array(movie_ratings)
        user_idx = np.argsort(sim_scores)
        sim_scores = sim_scores[user_idx][-neighbor_size:]
        movie_ratings = movie_ratings[user_idx][-neighbor_size:]
        prediction = np.dot(sim_scores,movie_ratings) / sim_scores.sum()
        prediction = prediction + rating_mean[user_id]
      else:
        prediction = rating_mean[user_id]
  else:
    prediction = rating_mean[user_id]
   
  return prediction

def RMSE2(y_true,y_pred):
  return np.sqrt(np.mean((np.array(y_true)-np.array(y_pred))**2))

Iteration : 10 ; Train RMSE = 0.9676 ; Test RMSE = 0.9688
Iteration : 20 ; Train RMSE = 0.9436 ; Test RMSE = 0.9490
Iteration : 30 ; Train RMSE = 0.9328 ; Test RMSE = 0.9412
Iteration : 40 ; Train RMSE = 0.9263 ; Test RMSE = 0.9370
Iteration : 50 ; Train RMSE = 0.9217 ; Test RMSE = 0.9346
Iteration : 60 ; Train RMSE = 0.9178 ; Test RMSE = 0.9329
Iteration : 70 ; Train RMSE = 0.9140 ; Test RMSE = 0.9315
Iteration : 80 ; Train RMSE = 0.9097 ; Test RMSE = 0.9302
Iteration : 90 ; Train RMSE = 0.9041 ; Test RMSE = 0.9286
Iteration : 100 ; Train RMSE = 0.8967 ; Test RMSE = 0.9264


In [6]:
# Hybrid 추천 알고리즘
def recommender_1(recom_list,mf):
  recommendations = np.array([
    mf.get_one_prediction(user,movie) for (user,movie) in recom_list
  ])
  return recommendations

def recommender_2(recom_list,neighbor_size=0):
  recommendations = np.array([
    CF_knn_bias(user,movie,neighbor_size) for (user,movie) in recom_list
  ])
  return recommendations

recom_list = np.array(ratings_test.iloc[:,[0,1]])

predictions_1 = recommender_1(recom_list,mf)
predictions_2 = recommender_2(recom_list,37)

print('recommendation 1 : ',RMSE2(ratings_test.iloc[:,2],predictions_1))
print('recommendation 2 : ',RMSE2(ratings_test.iloc[:,2],predictions_2))


weight = [0.8,0.2]
predictions = predictions_1 * weight[0] + predictions_2 * weight[1]

print('recommendation 1+2 : ',RMSE2(ratings_test.iloc[:,2],predictions))

recommendation 1 :  0.9263692393913044
recommendation 2 :  0.9290366558207726
recommendation 1+2 :  0.9229401668439577


In [ ]:
# 하이브리드 모델 최적화 시, 가중치를 조절하면서 테스트할 때 사용

result = []
weight_rate = []
for i in np.arange(0,1,0.01):
  weight = [i,1.0-i]
  predictions = predictions_1 * weight[0] + predictions_2 * weight[1]
  print("weights - %.2f :%.2f RMSE = %.7f"%(weight[0],weight[1],RMSE2(ratings_test.iloc[:,2],predictions)))
  result.append(RMSE2(ratings_test.iloc[:,2],predictions))
  weight_rate.append(weight)
print(min(result))
index_min = result.index(min(result))
print(weight_rate[index_min])